# Can patient characteristics explain or predict saved transfer benefit?

This notebook analyzes **patients**, using the completed three-seed results in Google Drive. **No forecasting models are trained, and no new RL-versus-TL experiment is run.** The outcome is the benefit already measured in those experiments: `RL MAE − TL MAE` (positive = transfer helped).

It examines the proposed claim in two parts:
1. Is greater **train-to-test shift** associated with **less benefit**?
2. Does that association weaken after accounting for **glucose-signal irregularity**, and does irregularity improve held-out-patient benefit prediction?

**Retrospective explanation and advance prediction are different.** Shift and test entropy can describe the completed evaluation. Only training-history features can be used for advance screening. Statistical adjustment cannot demonstrate causal mediation or “intrinsic” patient properties.

Use a **CPU Colab runtime**. This notebook is self-contained: no repository clone, GPU, raw XML files, or model checkpoints are required. Run all cells in order. It reads small saved CSVs and writes into a new timestamped Drive folder; existing results are preserved.


## 1. Mount Drive and set paths

Defaults match `run_on_colab.ipynb`. The inputs are:
- `bg-results/analysis/phase1_gru/patient_feature_table.csv` (or the LSTM/RNN equivalent);
- `bg-results/analysis/transfer_inference_{gru,lstm,rnn}/per_patient.csv`.

If a required table is missing, run the corresponding analysis stage in the original notebook; you do **not** need to retrain. Feature tables available for multiple architectures must agree. All 12 patients, three seeds, three architectures and four horizons are required to avoid selective reporting.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_RESULTS = Path('/content/drive/MyDrive/bg-results')
ANALYSIS_DIR = DRIVE_RESULTS / 'analysis'
PERMUTATIONS = 10_000
BOOTSTRAPS = 5_000
RANDOM_SEED = 20260923
# For a syntax/smoke check only, use 199 for both resampling counts.
# Such a run is labeled SMOKE in its output folder. Final inference needs the defaults.


## 2. Analysis choices and interpretation

This is an **exploratory follow-up designed after seeing earlier results**, not a preregistered confirmatory study.

- **Outcome:** saved absolute benefit in mg/dL; percentage benefit is a sensitivity analysis. GRU at 30 minutes is the reporting focus. All 12 architecture/horizon combinations are included.
- **Unit of analysis:** 12 patients. Outcomes already average the three matched seeds. We never count seeds or horizons as extra independent patients.
- **Association:** Spearman correlation with a two-sided patient-permutation test; Pearson correlation is also reported descriptively. The proposed shift claim requires a **negative** shift–benefit association.
- **Adjustment:** two small linear models: shift + test entropy, and shift + training entropy. Report standardized-predictor slopes, classical OLS t intervals/p-values, collinearity, and paired patient-bootstrap intervals for the change in absolute shift slope. OLS inference assumes linear, homoskedastic errors; inspect the patient plots and influence checks.
- **Prediction:** leave one patient out, fit on the other 11, and compare with their mean-benefit baseline. Three retrospective models use shift, test entropy, or both. Three history-only models use training entropy, training autocorrelation, or training entropy + training standard deviation. No hyperparameter search or outcome-driven feature selection is performed.
- **Multiple testing:** separate BH families for associations (72 tests), adjusted coefficients (96), and prediction scores (144), each including both outcomes and every architecture/horizon. Bootstrap intervals are descriptive, not simultaneous intervals.
- **Uncertainty:** conditional on the saved three-seed means; bootstrap patients, not windows. Shared population pretraining can couple outcomes between patients, so these cohort checks are not independent external validation. No new training-seed uncertainty is estimated.

A smaller adjusted shift slope **alone does not justify “largely explained by irregularity.”** Check whether the original association exists, whether entropy adds information, whether estimates depend on one patient, and whether prediction improves outside the fitting sample. A null shift association cannot be rescued by describing its attenuation as an explanation.


In [ ]:
import sys, subprocess, os
os.environ.setdefault('MPLCONFIGDIR', '/content/matplotlib-patient-benefit')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy>=1.24,<3', 'pandas>=2,<3', 'scipy>=1.10,<2', 'matplotlib>=3.7,<4'], check=True)


## 3. Load the analysis implementation

The implementation is embedded below to make this notebook runnable before the repository changes are pushed. Its source is also maintained as `RUN/experiments/run_patient_benefit_analysis.py`. Outputs include its SHA-256 hash and input-file hashes.


In [ ]:
RUNNER_SOURCE = '#!/usr/bin/env python3\n"""Patient-level explanation/prediction of SAVED transfer benefits; no training.\n\nAlso embedded in notebooks/analyze_patient_benefit_on_colab.ipynb so that the\nnotebook does not require a new repository checkout on Colab.\n"""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport platform\n\nimport numpy as np\nimport pandas as pd\nimport scipy\nfrom scipy.stats import rankdata, t\n\nCOHORT = [540, 544, 552, 559, 563, 567, 570, 575, 584, 588, 591, 596]\nMODELS = [\'gru\', \'lstm\', \'rnn\']\nHORIZONS = [15, 30, 45, 60]\nFEATURES = [\'shift_score\', \'sample_entropy\', \'train_sample_entropy\',\n            \'train_autocorr_lag1\', \'train_std\']\nSCREENS = {\n    \'retrospective_shift\': [\'shift_score\'],\n    \'retrospective_entropy\': [\'sample_entropy\'],\n    \'retrospective_shift_entropy\': [\'shift_score\', \'sample_entropy\'],\n    \'history_entropy\': [\'train_sample_entropy\'],\n    \'history_autocorrelation\': [\'train_autocorr_lag1\'],\n    \'history_entropy_std\': [\'train_sample_entropy\', \'train_std\'],\n}\n\n\ndef bh(p):\n    p = np.asarray(p, float)\n    if not np.isfinite(p).all():\n        raise ValueError(\'Cannot correct missing p-values; inspect degenerate fits.\')\n    order = np.argsort(p)\n    q = np.empty(len(p))\n    q[order] = np.minimum.accumulate(\n        (p[order] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]\n    return np.minimum(q, 1)\n\n\ndef corr(x, y):\n    a, b = np.asarray(x) - np.mean(x), np.asarray(y) - np.mean(y)\n    denominator = np.linalg.norm(a) * np.linalg.norm(b)\n    return float(a @ b / denominator) if denominator > 1e-12 else np.nan\n\n\ndef design(x):\n    x = np.asarray(x, float)\n    if x.ndim == 1:\n        x = x[:, None]\n    sd = x.std(axis=0)\n    if np.any(sd < 1e-12):\n        raise ValueError(\'Constant predictor\')\n    return np.column_stack([np.ones(len(x)), (x - x.mean(axis=0)) / sd])\n\n\ndef ols(x, y):\n    X = design(x)\n    if np.linalg.matrix_rank(X) != X.shape[1]:\n        raise ValueError(\'Collinear predictors\')\n    beta = np.linalg.lstsq(X, y, rcond=None)[0]\n    resid = y - X @ beta\n    df = len(y) - X.shape[1]\n    se = np.sqrt(np.diag(np.linalg.inv(X.T @ X)) * (resid @ resid) / df)\n    stat = np.divide(beta, se, out=np.zeros_like(beta), where=se > 0)\n    p = 2 * t.sf(np.abs(stat), df)\n    ci = np.column_stack([beta - t.ppf(.975, df) * se,\n                          beta + t.ppf(.975, df) * se])\n    return beta, p, ci\n\n\ndef loo_operator(x):\n    """Linear OLS prediction operator; each row excludes that patient\'s outcome.\n\n    Full-rank OLS predictions are invariant to affine predictor scaling, so\n    standardization here does not learn information used in a held-out fit.\n    Each fold is explicitly refit and checked, rather than using fitted error.\n    """\n    X = design(x)\n    n = len(X)\n    op = np.zeros((n, n))\n    for i in range(n):\n        keep = np.arange(n) != i\n        if np.linalg.matrix_rank(X[keep]) < X.shape[1]:\n            raise ValueError(\'Singular leave-one-patient-out fold\')\n        op[i, keep] = X[i] @ np.linalg.pinv(X[keep])\n    return op\n\n\ndef validate_inputs(features, outcomes):\n    if features.patient_id.duplicated().any() or sorted(features.patient_id) != COHORT:\n        raise ValueError(\'Feature table must contain each of the 12 patients once.\')\n    if not np.isfinite(features[FEATURES].to_numpy(float)).all():\n        raise ValueError(\'Missing/nonfinite patient features: do not silently impute/drop patients.\')\n    keys = [\'model\', \'horizon_minutes\', \'patient_id\']\n    if outcomes.duplicated(keys).any():\n        raise ValueError(\'Duplicate patient outcomes (do not concatenate seed rows).\')\n    expected = {(m, h, p) for m in MODELS for h in HORIZONS for p in COHORT}\n    if set(map(tuple, outcomes[keys].to_numpy())) != expected:\n        raise ValueError(\'Expected exactly 12 patients x 3 models x 4 horizons.\')\n    if not (outcomes.n_matched_seeds == 3).all():\n        raise ValueError(\'All saved patient outcomes must average three matched seeds.\')\n    values = outcomes[[\'regular_mae_mg_dl\', \'transfer_mae_mg_dl\']].to_numpy(float)\n    if not np.isfinite(values).all() or np.any(values <= 0):\n        raise ValueError(\'Invalid saved MAE values.\')\n    if \'tl_minus_rl_mae_mg_dl\' in outcomes:\n        np.testing.assert_allclose(outcomes.tl_minus_rl_mae_mg_dl,\n                                   values[:, 1] - values[:, 0], atol=1e-9)\n\n\ndef load_inputs(root):\n    root = Path(root)\n    sources = []\n    # Explicit paths prevent accidentally selecting stale seedwise experiments.\n    feature_paths = [root / f\'phase1_{m}\' / \'patient_feature_table.csv\' for m in MODELS]\n    existing = [p for p in feature_paths if p.is_file()]\n    if not existing:\n        raise FileNotFoundError(f\'No phase1_*/patient_feature_table.csv in {root}. \'\n                                \'Run the existing notebook shift/signal analysis first.\')\n    features = pd.read_csv(existing[0]).sort_values(\'patient_id\').reset_index(drop=True)\n    sources.append(existing[0])\n    for path in existing[1:]:\n        other = pd.read_csv(path).sort_values(\'patient_id\').reset_index(drop=True)\n        np.testing.assert_array_equal(features.patient_id, other.patient_id)\n        np.testing.assert_allclose(features[FEATURES], other[FEATURES], rtol=1e-8)\n        sources.append(path)\n    frames = []\n    for model in MODELS:\n        path = root / f\'transfer_inference_{model}\' / \'per_patient.csv\'\n        if not path.is_file():\n            raise FileNotFoundError(f\'{path}: run the existing transfer analysis stage first.\')\n        frame = pd.read_csv(path)\n        frame[\'model\'] = frame.model.str.lower()\n        if set(frame.model) != {model}:\n            raise ValueError(f\'Model mismatch in {path}\')\n        frames.append(frame)\n        sources.append(path)\n    outcomes = pd.concat(frames, ignore_index=True)\n    validate_inputs(features, outcomes)\n    outcomes[\'benefit_mg_dl\'] = outcomes.regular_mae_mg_dl - outcomes.transfer_mae_mg_dl\n    outcomes[\'benefit_pct\'] = 100 * outcomes.benefit_mg_dl / outcomes.regular_mae_mg_dl\n    return features, outcomes, sources\n\n\ndef analyze(features, outcomes, permutations=10000, bootstraps=5000, seed=20260923):\n    """Keep all architecture/horizon outcomes aligned to the same patient draws."""\n    if permutations < 99 or bootstraps < 99:\n        raise ValueError(\'Use at least 99 resamples even for a smoke check.\')\n    rng = np.random.default_rng(seed)\n    n = len(COHORT)\n    perms = np.array([rng.permutation(n) for _ in range(permutations)])\n    draws = rng.integers(0, n, size=(bootstraps, n))\n    f = features.set_index(\'patient_id\').loc[COHORT]\n    operators = {name: loo_operator(f[cols]) for name, cols in SCREENS.items()}\n    baseline = (np.ones((n, n)) - np.eye(n)) / (n - 1)\n    associations, adjusted, attenuation, cv, predictions, influence = [], [], [], [], [], []\n    for (model, horizon), group in outcomes.groupby([\'model\', \'horizon_minutes\']):\n        group = group.set_index(\'patient_id\').loc[COHORT]\n        print(f\'Analyzing {model.upper()} {horizon} min (12 patients)\', flush=True)\n        for outcome in [\'benefit_mg_dl\', \'benefit_pct\']:\n            y = group[outcome].to_numpy(float)\n            if y.std() < 1e-12:\n                raise ValueError(\'Constant benefit: association cannot be estimated.\')\n            tag = dict(model=model, horizon_minutes=int(horizon), outcome=outcome, n_patients=n)\n            yrank = rankdata(y)\n            for col in [\'shift_score\', \'sample_entropy\', \'train_sample_entropy\']:\n                x = f[col].to_numpy(float)\n                xr = rankdata(x) - rankdata(x).mean()\n                yc = yrank - yrank.mean()\n                rho = corr(xr, yc)\n                null = yc[perms] @ xr / (np.linalg.norm(xr) * np.linalg.norm(yc))\n                p = (1 + np.count_nonzero(abs(null) >= abs(rho) - 1e-12)) / (permutations + 1)\n                associations.append({**tag, \'feature\': col, \'spearman_rho\': rho,\n                                     \'pearson_r\': corr(x, y), \'permutation_p\': p})\n                for i, pid in enumerate(COHORT):\n                    keep = np.arange(n) != i\n                    influence.append({**tag, \'feature\': col, \'omitted_patient\': pid,\n                                      \'spearman_rho\': corr(rankdata(x[keep]), rankdata(y[keep]))})\n            # Standardized X slopes have outcome units per one predictor SD.\n            # Adjustment describes shared association, never causal mediation.\n            shift = f.shift_score.to_numpy(float)\n            unadjusted = ols(shift, y)[0][1]\n            for entropy in [\'sample_entropy\', \'train_sample_entropy\']:\n                x = f[[\'shift_score\', entropy]].to_numpy(float)\n                beta, pvalues, ci = ols(x, y)\n                for j, col in enumerate([\'shift_score\', entropy], 1):\n                    adjusted.append({**tag, \'adjust_for\': entropy, \'term\': col,\n                                     \'coefficient_per_sd\': beta[j], \'ci95_low\': ci[j, 0],\n                                     \'ci95_high\': ci[j, 1], \'ols_t_p\': pvalues[j]})\n                boot = []\n                for index in draws:\n                    try:\n                        X = design(x[index])\n                        if np.linalg.matrix_rank(X) != 3:\n                            continue\n                        a = np.linalg.lstsq(design(shift[index]), y[index], rcond=None)[0][1]\n                        b = np.linalg.lstsq(X, y[index], rcond=None)[0][1]\n                        boot.append([a, b, abs(a) - abs(b)])\n                    except ValueError:\n                        continue\n                if len(boot) < .9 * bootstraps:\n                    raise ValueError(\'Too many singular patient bootstrap samples.\')\n                boot = np.array(boot)\n                bounds = np.quantile(boot, [.025, .975], axis=0)\n                attenuation.append({**tag, \'adjust_for\': entropy,\n                    \'shift_unadjusted\': unadjusted, \'shift_adjusted\': beta[1],\n                    \'absolute_slope_reduction\': abs(unadjusted) - abs(beta[1]),\n                    \'reduction_ci95_low\': bounds[0, 2], \'reduction_ci95_high\': bounds[1, 2],\n                    \'unadjusted_ci95_low\': bounds[0, 0], \'unadjusted_ci95_high\': bounds[1, 0],\n                    \'adjusted_ci95_low\': bounds[0, 1], \'adjusted_ci95_high\': bounds[1, 1],\n                    \'shift_entropy_r\': corr(shift, x[:, 1]),\n                    \'vif\': 1 / (1 - corr(shift, x[:, 1]) ** 2),\n                    \'valid_bootstraps\': len(boot)})\n            base = baseline @ y\n            base_mse = np.mean((y - base) ** 2)\n            yp = y[perms]\n            null_base_mse = np.mean((yp - yp @ baseline.T) ** 2, axis=1)\n            for name, op in operators.items():\n                pred = op @ y\n                mse = np.mean((y - pred) ** 2)\n                skill = 1 - mse / base_mse\n                null_skill = 1 - np.mean((yp - yp @ op.T) ** 2, axis=1) / null_base_mse\n                p = (1 + np.count_nonzero(null_skill >= skill - 1e-12)) / (permutations + 1)\n                cv.append({**tag, \'screen\': name, \'available_before_test\': name.startswith(\'history_\'),\n                           \'rmse\': np.sqrt(mse), \'baseline_rmse\': np.sqrt(base_mse),\n                           \'mae\': np.mean(abs(y - pred)), \'mse_skill\': skill,\n                           \'permutation_p\': p})\n                for i, pid in enumerate(COHORT):\n                    predictions.append({**tag, \'screen\': name, \'patient_id\': pid,\n                                        \'observed\': y[i], \'predicted\': pred[i], \'baseline\': base[i]})\n    cv_table = pd.DataFrame(cv)\n    increments = []\n    for (model, horizon, outcome), group in cv_table.groupby([\'model\', \'horizon_minutes\', \'outcome\']):\n        indexed = group.set_index(\'screen\')\n        joint_mse = indexed.loc[\'retrospective_shift_entropy\', \'rmse\'] ** 2\n        for reference in [\'retrospective_shift\', \'retrospective_entropy\']:\n            ref_mse = indexed.loc[reference, \'rmse\'] ** 2\n            increments.append(dict(model=model, horizon_minutes=int(horizon), outcome=outcome,\n                joint_model=\'retrospective_shift_entropy\', reference_model=reference,\n                held_out_mse_reduction=ref_mse - joint_mse,\n                relative_mse_reduction=1 - joint_mse / ref_mse))\n    tables = {\'associations\': pd.DataFrame(associations), \'adjusted_models\': pd.DataFrame(adjusted),\n              \'attenuation\': pd.DataFrame(attenuation), \'prediction_scores\': pd.DataFrame(cv),\n              \'held_out_predictions\': pd.DataFrame(predictions), \'leave_one_out_influence\': pd.DataFrame(influence),\n              \'incremental_prediction\': pd.DataFrame(increments)}\n    # Broad families include BOTH benefit definitions, ALL 12 cells and every\n    # candidate feature/model. No favorable horizon/model is silently selected.\n    for name, p in [(\'associations\', \'permutation_p\'), (\'adjusted_models\', \'ols_t_p\'),\n                    (\'prediction_scores\', \'permutation_p\')]:\n        tables[name][\'bh_q\'] = bh(tables[name][p])\n    return tables\n\n\ndef save_outputs(out, features, outcomes, tables, sources, config):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    features.to_csv(out / \'patient_features.csv\', index=False)\n    outcomes.to_csv(out / \'saved_patient_benefits.csv\', index=False)\n    for name, frame in tables.items():\n        frame.to_csv(out / f\'{name}.csv\', index=False)\n    manifest = {**config, \'python\': platform.python_version(), \'numpy\': np.__version__,\n                \'pandas\': pd.__version__, \'scipy\': scipy.__version__, \'sources\': [\n                    {\'path\': str(p), \'sha256\': hashlib.sha256(Path(p).read_bytes()).hexdigest()}\n                    for p in sources], \'runner_sha256\': hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n                \'independent_patients\': 12, \'forecast_models_trained\': 0,\n                \'uncertainty\': \'patient-level, conditional on saved three-seed means; no seed resampling\',\n                \'families\': {k: len(tables[k]) for k in [\'associations\', \'adjusted_models\', \'prediction_scores\']}}\n    (out / \'manifest.json\').write_text(json.dumps(manifest, indent=2) + \'\\n\')\n    def focus(frame):\n        return frame[(frame.model == \'gru\') & (frame.horizon_minutes == 30) &\n                     (frame.outcome == \'benefit_mg_dl\')]\n    a = focus(tables[\'associations\'])\n    c = focus(tables[\'prediction_scores\'])\n    shift = a[a.feature == \'shift_score\'].iloc[0]\n    lines = [\'# Patient-level transfer-benefit analysis\', \'\',\n             \'Exploratory follow-up using existing outcomes; no forecasting model was trained.\',\n             \'Positive benefit = saved RL MAE minus saved TL MAE. GRU-30 is the reporting focus, not a preregistered primary test.\', \'\',\n             \'## GRU, 30 minutes, absolute benefit\', \'\',\n             f"Shift Spearman rho = {shift.spearman_rho:.3f}; permutation p = {shift.permutation_p:.4f}; BH q = {shift.bh_q:.4f}.",\n             \'The original direction requires a NEGATIVE shift-benefit association.\', \'\',\n             \'Prediction MSE skill relative to leave-one-patient-out mean-benefit baseline:\']\n    for r in c.itertuples():\n        lines.append(f\'- {r.screen}: {100*r.mse_skill:+.1f}%; permutation BH q={r.bh_q:.4f}.\')\n    lines += [\'\', \'## How to interpret the files\', \'\',\n        \'- associations.csv: shift/entropy associations, all cells and both absolute and percentage benefit.\',\n        \'- adjusted_models.csv: shift and entropy entered together, with classical OLS t intervals/p-values (linear, homoskedastic error assumptions).\',\n        \'- attenuation.csv: reduction in absolute standardized shift slope after entropy adjustment, paired patient-bootstrap interval, and collinearity. Negative reduction means a larger adjusted slope. No percentage attenuation is reported because near-zero slopes make it unstable.\',\n        \'- prediction_scores.csv: fully held-out patient outcomes; history_* models use training features only. Retrospective models use test-derived features and cannot establish advance prediction.\',\n        \'- incremental_prediction.csv: does the joint shift/entropy model predict held-out benefit better than either alone? Positive MSE reduction means improvement; these comparisons are descriptive, not additional significance tests.\',\n        \'- leave_one_out_influence.csv: how much each individual patient changes the association.\', \'\',\n        \'## Limits on the claim\', \'\',\n        \'A smaller adjusted shift slope alone does not establish that irregularity explains the relationship. Examine the original association, entropy coefficient, uncertainty, prediction performance and leave-one-patient-out sensitivity together. A null shift association cannot be rescued by calling its attenuation an explanation.\',\n        \'These data cannot establish intrinsic irregularity or causal mediation. Sample entropy describes the observed segment. Horizons, architectures and seeds reuse the same 12 patients and are not independent replications.\',\n        \'Patient bootstrap/permutation results assume patient-level exchangeability, conditional on the fitted models. Shared population pretraining can couple patient outcomes; these are exploratory cohort checks, not independent external validation. The bootstrap does not include training-seed uncertainty.\',\n        \'BH correction is separate for association, adjusted-coefficient and prediction families, each spanning both outcomes and all configurations. Bootstrap intervals and influence checks are descriptive, without simultaneous coverage.\',\n        \'This follow-up was designed after seeing earlier results. It must not be presented as preregistered confirmation, even if a p-value is small.\', \'\']\n    (out / \'REPORT.md\').write_text(\'\\n\'.join(lines))\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    g = outcomes[(outcomes.model == \'gru\') & (outcomes.horizon_minutes == 30)].merge(features, on=\'patient_id\')\n    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), constrained_layout=True)\n    for ax, col, title in zip(axes, [\'shift_score\', \'sample_entropy\', \'train_sample_entropy\'],\n                              [\'Train-to-test shift (retrospective)\', \'Test entropy (retrospective)\', \'Training entropy (available beforehand)\']):\n        ax.scatter(g[col], g.benefit_mg_dl)\n        for r in g.itertuples():\n            ax.annotate(str(r.patient_id), (getattr(r, col), r.benefit_mg_dl), fontsize=7, xytext=(3, 3), textcoords=\'offset points\')\n        ax.axhline(0, color=\'gray\', linewidth=.7)\n        ax.set(xlabel=title, ylabel=\'Saved transfer benefit (mg/dL)\')\n    fig.suptitle(\'GRU, 30 minutes: patient-level saved benefit (three-seed means)\')\n    fig.savefig(out / \'patient_benefit_diagnostics.png\', dpi=180)\n    plt.close(fig)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--analysis-dir\', required=True)\n    parser.add_argument(\'--output-dir\', required=True)\n    parser.add_argument(\'--permutations\', type=int, default=10000)\n    parser.add_argument(\'--bootstraps\', type=int, default=5000)\n    parser.add_argument(\'--seed\', type=int, default=20260923)\n    args = parser.parse_args()\n    out = Path(args.output_dir)\n    if out.exists() and any(out.iterdir()):\n        raise FileExistsError(\'Choose a new output folder; completed analyses are not overwritten.\')\n    features, outcomes, sources = load_inputs(args.analysis_dir)\n    tables = analyze(features, outcomes, args.permutations, args.bootstraps, args.seed)\n    save_outputs(out, features, outcomes, tables, sources, vars(args))\n    print(f\'Saved report, tables, provenance and figure in {out}\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
RUNNER_PATH = Path('/content/run_patient_benefit_analysis.py')
RUNNER_PATH.write_text(RUNNER_SOURCE)
import importlib.util
spec = importlib.util.spec_from_file_location('patient_benefit', RUNNER_PATH)
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
print('Patient-analysis implementation ready; no model training code is invoked.')


## 4. Validate and snapshot inputs

Only the relevant CSVs are copied from Drive. This checks cohort completeness, three-seed aggregation, finite features, duplicate rows, outcome signs, and agreement between available feature tables. The exact input snapshot and protocol are archived before analysis.


In [ ]:
from datetime import datetime, timezone
import shutil, json, hashlib
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
smoke = PERMUTATIONS < 10_000 or BOOTSTRAPS < 5_000
run_name = ('SMOKE_' if smoke else '') + stamp
LOCAL_RUN = Path('/content/patient_benefit_analysis') / run_name
DRIVE_RUN = ANALYSIS_DIR / 'patient_benefit_followup' / run_name
INPUT_DIR = LOCAL_RUN / 'inputs'
INPUT_DIR.mkdir(parents=True, exist_ok=False)
source_paths = []
for model in analysis.MODELS:
    source_paths.append(ANALYSIS_DIR / f'transfer_inference_{model}' / 'per_patient.csv')
    feature = ANALYSIS_DIR / f'phase1_{model}' / 'patient_feature_table.csv'
    if feature.is_file():
        source_paths.append(feature)
for source in source_paths:
    if not source.is_file():
        raise FileNotFoundError(f'Missing {source}. Run the saved transfer-analysis stage first.')
    target = INPUT_DIR / source.relative_to(ANALYSIS_DIR)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
features, outcomes, sources = analysis.load_inputs(INPUT_DIR)
PROTOCOL = dict(permutations=PERMUTATIONS, bootstraps=BOOTSTRAPS, random_seed=RANDOM_SEED,
    smoke=smoke, reporting_focus='GRU 30 minutes; absolute benefit',
    status='exploratory, post hoc; all 12 cells reported',
    units='12 patients; three-seed means; no forecasting training',
    outcome='regular_mae_mg_dl - transfer_mae_mg_dl',
    screens=analysis.SCREENS,
    multiplicity='BH separately over all association, adjusted-coefficient, and CV tests; both outcomes',
    source_drive_paths=[str(p) for p in source_paths])
(LOCAL_RUN / 'protocol.json').write_text(json.dumps(PROTOCOL, indent=2))
shutil.copy2(RUNNER_PATH, LOCAL_RUN / RUNNER_PATH.name)
shutil.copytree(LOCAL_RUN, DRIVE_RUN)
print(f'Validated {len(features)} patients and {len(outcomes)} saved patient/configuration outcomes.')
print(f'Archive: {DRIVE_RUN}')
display(features[['patient_id'] + analysis.FEATURES])


## 5. Run patient analyses

The notebook prints progress for each architecture/horizon. This is CPU statistical analysis, not another multi-hour training grid. Patient bootstrap draws are reused across configurations to preserve alignment. If interrupted, rerun this cell; only this new run's analysis files are refreshed.


In [ ]:
import time
started = time.perf_counter()
OUTPUT_DIR = LOCAL_RUN / 'outputs'
try:
    tables = analysis.analyze(features, outcomes, PERMUTATIONS, BOOTSTRAPS, RANDOM_SEED)
    analysis.save_outputs(OUTPUT_DIR, features, outcomes, tables, sources, PROTOCOL)
finally:
    shutil.copytree(LOCAL_RUN, DRIVE_RUN, dirs_exist_ok=True)
print(f'Finished in {(time.perf_counter() - started)/60:.1f} minutes.')
print(f'Results: {DRIVE_RUN / "outputs"}')


## 6. Read the focused results and patient diagnostics

`mse_skill > 0` means better held-out prediction than the mean-benefit baseline; negative skill means worse. The RMSE here measures errors in predicting **patient benefit**, not glucose forecasts. “Retrospective” prediction is still unavailable before the test period even though the patient's benefit was held out.


In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown((OUTPUT_DIR / 'REPORT.md').read_text()))
for name in ['associations', 'adjusted_models', 'attenuation', 'prediction_scores', 'incremental_prediction']:
    frame = tables[name]
    selected = frame[(frame.model == 'gru') & (frame.horizon_minutes == 30) &
                     (frame.outcome == 'benefit_mg_dl')]
    print(name)
    display(selected)
display(Image(filename=str(OUTPUT_DIR / 'patient_benefit_diagnostics.png')))


## 7. Check consistency across all settings

The same patients appear in all settings: agreement is a robustness description, **not 12 independent confirmations**. Inspect percentage-benefit sensitivity and individual-patient influence before choosing wording for the article.

**Claim boundaries:**
- A robust negative shift association supports a *retrospective* association with less benefit.
- Adjustment and better held-out performance may suggest shared information with irregularity; they do not establish a causal explanation.
- Only the `history_*` models can support advance screening, and this cohort remains exploratory.
- If associations are inconsistent or prediction is worse than the baseline, retain that result instead of searching for a favorable subset.


In [ ]:
for outcome in ['benefit_mg_dl', 'benefit_pct']:
    a = tables['associations']
    a = a[(a.feature == 'shift_score') & (a.outcome == outcome)]
    print(outcome, '— shift association in every architecture/horizon')
    display(a[['model', 'horizon_minutes', 'spearman_rho', 'permutation_p', 'bh_q']])
    c = tables['prediction_scores']
    c = c[(c.outcome == outcome) & c.available_before_test]
    display(c.pivot(index=['model', 'horizon_minutes'], columns='screen', values='mse_skill'))
print('Files to inspect in Drive:')
for p in sorted((DRIVE_RUN / 'outputs').iterdir()):
    print(p.name)
print('The article has not been edited by this notebook.')
